<a href="https://colab.research.google.com/github/Faisaleka21/Machine_Learning/blob/main/UAS_DM_Aprori.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================
# 1. IMPORT LIBRARY
# =====================================================

import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
import warnings
warnings.filterwarnings('ignore')

print("Library berhasil diimport!")

Library berhasil diimport!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# =====================================================
# 2. MEMBACA DATASET
# =====================================================

# Baca dataset dari file
df = pd.read_csv('https://raw.githubusercontent.com/Faisaleka21/dataset/refs/heads/main/BreadBasket_DMS-ASOSIASI.csv',sep=';')

print("\n=== INFORMASI DATASET ===")
print(f"Jumlah baris : {df.shape[0]}")
print(f"Jumlah kolom : {df.shape[1]}")
print(f"Nama kolom : {list(df.columns)}")
print(f"Tipe data :")
print(df.dtypes)

print("\n=== 5 DATA PERTAMA ===")
print(df.head(5).to_string(index=True))


=== INFORMASI DATASET ===
Jumlah baris : 21293
Jumlah kolom : 4
Nama kolom : ['Date', 'Time', 'Transaction', 'Item']
Tipe data :
Date           object
Time           object
Transaction     int64
Item           object
dtype: object

=== 5 DATA PERTAMA ===
         Date      Time  Transaction           Item
0  30/10/2016  09:58:11            1          Bread
1  30/10/2016  10:05:34            2   Scandinavian
2  30/10/2016  10:05:34            2   Scandinavian
3  30/10/2016  10:07:57            3  Hot chocolate
4  30/10/2016  10:07:57            3            Jam


In [ ]:
# =====================================================
# 3. PREPROCESSING DATA
# =====================================================

# Pemeriksaan missing value
missing_values = df.isnull().sum()
print("\n=== CEK MISSING VALUE ===")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "Tidak ada missing value")

# Pemeriksaan data duplikat
duplicate_count = df.duplicated().sum()
print(f"\n=== CEK DATA DUPLIKAT ===")
print(f"Jumlah data duplikat: {duplicate_count}")

# Hapus data duplikat jika ada
if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Data duplikat berhasil dihapus!")

# Kelompokkan item berdasarkan Transaction
basket = df.groupby(['Transaction'])['Item'].apply(list).reset_index()

print("\n=== HASIL PREPROCESSING ===")
print(f"Jumlah transaksi : {len(basket)}")

# One-hot encoding
items = set()
for items_list in basket['Item']:
    items.update(items_list)

items = sorted(list(items))
print(f"Jumlah item unik : {len(items)}")

# Membuat one-hot encoding
one_hot = pd.DataFrame(0, index=basket['Transaction'], columns=items)

for idx, row in basket.iterrows():
    for item in row['Item']:
        if item in one_hot.columns:
            one_hot.loc[row['Transaction'], item] = 1

print(f"Ukuran data hasil encoding : {one_hot.shape}")

print("\n=== 5 DATA PERTAMA HASIL ENCODING ===")
print(one_hot.head(5).to_string(index=True))


=== CEK MISSING VALUE ===
Tidak ada missing value

=== CEK DATA DUPLIKAT ===
Jumlah data duplikat: 1653
Data duplikat berhasil dihapus!

=== HASIL PREPROCESSING ===
Jumlah transaksi : 9531
Jumlah item unik : 95
Ukuran data hasil encoding : (9531, 95)

=== 5 DATA PERTAMA HASIL ENCODING ===
             Adjustment  Afternoon with the baker  Alfajores  Argentina Night  Art Tray  Bacon  Baguette  Bakewell  Bare Popcorn  Basket  Bowl Nic Pitt  Bread  Bread Pudding  Brioche and salami  Brownie  Cake  Caramel bites  Cherry me Dried fruit  Chicken Stew  Chicken sand  Chimichurri Oil  Chocolates  Christmas common  Coffee  Coffee granules   Coke  Cookies  Crepes  Crisps  Drinking chocolate spoons   Duck egg  Dulce de Leche  Eggs  Ella's Kitchen Pouches  Empanadas  Extra Salami or Feta  Fairy Doors  Farm House  Focaccia  Frittata  Fudge  Gift voucher  Gingerbread syrup  Granola  Hack the stack  Half slice Monster   Hearty & Seasonal  Honey  Hot chocolate  Jam  Jammie Dodgers  Juice  Keeping It L

In [ ]:
# =====================================================
# 4. ALGORITMA APRIORI
# =====================================================

min_support = 0.02  #ini nilai untuk supportnya

# Cari frequent itemsets
frequent_itemsets = apriori(one_hot, min_support=min_support, use_colnames=True)

# Tambahkan kolom panjang itemset
frequent_itemsets['panjang_itemset'] = frequent_itemsets['itemsets'].apply(len)

# Urutkan berdasarkan support terbesar
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False)

print("\n=== FREQUENT ITEMSETS ===")
print(f"Total frequent itemsets: {len(frequent_itemsets)}")
print("\nNo | Itemset | Support | Panjang Itemset")
print("-" * 70)

for i, row in enumerate(frequent_itemsets.head(20).itertuples(), 1):
    itemset_str = ', '.join(list(row.itemsets))
    print(f"{i:2} | {itemset_str:30} | {row.support:.4f} | {row.panjang_itemset:8}")


=== FREQUENT ITEMSETS ===
Total frequent itemsets: 36

No | Itemset | Support | Panjang Itemset
----------------------------------------------------------------------
 1 | Coffee                         | 0.4751 |        1
 2 | Bread                          | 0.3249 |        1
 3 | Tea                            | 0.1416 |        1
 4 | Cake                           | 0.1031 |        1
 5 | Bread, Coffee                  | 0.0894 |        2
 6 | Pastry                         | 0.0855 |        1
 7 | NONE                           | 0.0790 |        1
 8 | Sandwich                       | 0.0713 |        1
 9 | Medialuna                      | 0.0614 |        1
10 | Hot chocolate                  | 0.0579 |        1
11 | Cake, Coffee                   | 0.0543 |        2
12 | Cookies                        | 0.0540 |        1
13 | Tea, Coffee                    | 0.0495 |        2
14 | Pastry, Coffee                 | 0.0472 |        2
15 | NONE, Coffee                   | 0.0421 |  

In [ ]:
# =====================================================
# 5. ASSOCIATION RULES (PENENTU CONVIEDENCE)
# =====================================================

# Generate association rules
rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.5) # ini nilai conviedence

# Urutkan berdasarkan Lift terbesar
rules = rules.sort_values('lift', ascending=False)

print("\n=== ASSOCIATION RULES ===")
print(f"Total aturan yang dihasilkan: {len(rules)}")
print("\nNo | Jika Membeli | Maka Membeli | Support | Confidence | Lift")
print("-" * 100)

for i, row in enumerate(rules.head(20).itertuples(), 1):
    antecedents = ', '.join(list(row.antecedents))
    consequents = ', '.join(list(row.consequents))
    print(f"{i:2} | {antecedents:25} | {consequents:25} | {row.support:.4f} | {row.confidence:.4f} | {row.lift:.4f}")



=== ASSOCIATION RULES ===
Total aturan yang dihasilkan: 9

No | Jika Membeli | Maka Membeli | Support | Confidence | Lift
----------------------------------------------------------------------------------------------------
 1 | Toast                     | Coffee                    | 0.0235 | 0.7044 | 1.4827
 2 | Medialuna                 | Coffee                    | 0.0349 | 0.5692 | 1.1982
 3 | Pastry                    | Coffee                    | 0.0472 | 0.5521 | 1.1622
 4 | Juice                     | Coffee                    | 0.0205 | 0.5342 | 1.1245
 5 | NONE                      | Coffee                    | 0.0421 | 0.5325 | 1.1209
 6 | Sandwich                  | Coffee                    | 0.0380 | 0.5324 | 1.1206
 7 | Cake                      | Coffee                    | 0.0543 | 0.5270 | 1.1092
 8 | Cookies                   | Coffee                    | 0.0280 | 0.5184 | 1.0913
 9 | Hot chocolate             | Coffee                    | 0.0294 | 0.5072 | 1.0677


In [ ]:
# =====================================================
# 6. EVALUASI HASIL
# =====================================================

print("\n=== HASIL EVALUASI ===")
print(f"Rata-rata Support : {rules['support'].mean():.4f}")
print(f"Rata-rata Confidence : {rules['confidence'].mean():.4f}")
print(f"Rata-rata Lift : {rules['lift'].mean():.4f}")

print("\nInterpretasi:")
print("* Lift > 1 menunjukkan hubungan positif antar item.")
print("* Confidence tinggi menunjukkan aturan yang kuat.")
print("* Support tinggi menunjukkan kombinasi item sering muncul.")


=== HASIL EVALUASI ===
Rata-rata Support : 0.0353
Rata-rata Confidence : 0.5531
Rata-rata Lift : 1.1641

Interpretasi:
* Lift > 1 menunjukkan hubungan positif antar item.
* Confidence tinggi menunjukkan aturan yang kuat.
* Support tinggi menunjukkan kombinasi item sering muncul.


In [ ]:
# =====================================================
# 7. ATURAN TERBAIK
# =====================================================

print("\n=== 10 ATURAN ASOSIASI TERBAIK (LIFT TERTINGGI) ===")

# Membuat tabel aturan terbaik
aturan_terbaik = rules.head(10).copy()

# Mengubah frozenset menjadi string
aturan_terbaik['Jika Membeli'] = aturan_terbaik['antecedents'].apply(lambda x: ', '.join(list(x)))
aturan_terbaik['Maka Membeli'] = aturan_terbaik['consequents'].apply(lambda x: ', '.join(list(x)))

# Memilih kolom yang ingin ditampilkan
tabel_aturan = aturan_terbaik[['Jika Membeli',
                               'Maka Membeli',
                               'support',
                               'confidence',
                               'lift']]

# Mengubah nama kolom
tabel_aturan.columns = ['Jika Membeli',
                         'Maka Membeli',
                         'Support',
                         'Confidence',
                         'Lift']

# Menampilkan tabel
display(tabel_aturan)


=== 10 ATURAN ASOSIASI TERBAIK (LIFT TERTINGGI) ===


,Jika Membeli,Maka Membeli,Support,Confidence,Lift
7,Toast,Coffee,0.023502,0.704403,1.482699
4,Medialuna,Coffee,0.034939,0.569231,1.198175
1,Pastry,Coffee,0.047214,0.552147,1.162216
8,Juice,Coffee,0.020460,0.534247,1.124537
2,NONE,Coffee,0.042073,0.532537,1.120938
3,Sandwich,Coffee,0.037981,0.532353,1.120551
0,Cake,Coffee,0.054349,0.526958,1.109196
6,Cookies,Coffee,0.028014,0.518447,1.091280
5,Hot chocolate,Coffee,0.029378,0.507246,1.067704


In [ ]:
# =====================================================
# 8. KESIMPULAN
# =====================================================

print("\n=== KESIMPULAN ===")

# Analisis kesimpulan otomatis
total_rules = len(rules)
avg_support = rules['support'].mean()
avg_confidence = rules['confidence'].mean()
avg_lift = rules['lift'].mean()

print(f"Berdasarkan analisis data transaksi dengan algoritma Apriori:")

if total_rules > 0:
    print(f"1. Ditemukan {total_rules} aturan asosiasi dengan minimum support {min_support} dan minimum confidence 0.5.")

    if avg_lift > 1:
        print("2. Rata-rata Lift > 1 ({:.4f}) menunjukkan bahwa mayoritas aturan memiliki hubungan positif antar item.".format(avg_lift))
    else:
        print("2. Rata-rata Lift ≤ 1 ({:.4f}) menunjukkan bahwa mayoritas aturan tidak memiliki hubungan positif yang kuat.".format(avg_lift))

    if avg_confidence > 0.7:
        print("3. Rata-rata Confidence tinggi ({:.4f}) menunjukkan aturan yang cukup kuat.".format(avg_confidence))
    else:
        print("3. Rata-rata Confidence sedang ({:.4f}) menunjukkan aturan yang cukup baik.".format(avg_confidence))

    if avg_support > 0.05:
        print("4. Rata-rata Support tinggi ({:.4f}) menunjukkan kombinasi item sering muncul dalam transaksi.".format(avg_support))
    else:
        print("4. Rata-rata Support ({:.4f}) menunjukkan kombinasi item jarang muncul dalam transaksi.".format(avg_support))

    # Aturan terbaik
    best_rule = rules.iloc[0]
    ante = ', '.join(list(best_rule['antecedents']))
    cons = ', '.join(list(best_rule['consequents']))
    print(f"\n5. Aturan terbaik: Jika membeli '{ante}' maka membeli '{cons}' dengan Lift {best_rule['lift']:.4f}.")
    print(f"   (Support: {best_rule['support']:.4f}, Confidence: {best_rule['confidence']:.4f})")
else:
    print("Tidak ditemukan aturan asosiasi yang memenuhi minimum support dan minimum confidence yang ditentukan.")

print("\nKesimpulan ini memberikan gambaran umum tentang pola pembelian pelanggan.")
print("Aturan dengan Lift > 1 menunjukkan bahwa pembelian suatu item meningkatkan kemungkinan pembelian item lain.")


=== KESIMPULAN ===
Berdasarkan analisis data transaksi dengan algoritma Apriori:
1. Ditemukan 9 aturan asosiasi dengan minimum support 0.02 dan minimum confidence 0.5.
2. Rata-rata Lift > 1 (1.1641) menunjukkan bahwa mayoritas aturan memiliki hubungan positif antar item.
3. Rata-rata Confidence sedang (0.5531) menunjukkan aturan yang cukup baik.
4. Rata-rata Support (0.0353) menunjukkan kombinasi item jarang muncul dalam transaksi.

5. Aturan terbaik: Jika membeli 'Toast' maka membeli 'Coffee' dengan Lift 1.4827.
   (Support: 0.0235, Confidence: 0.7044)

Kesimpulan ini memberikan gambaran umum tentang pola pembelian pelanggan.
Aturan dengan Lift > 1 menunjukkan bahwa pembelian suatu item meningkatkan kemungkinan pembelian item lain.
